### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
working_folder='./'

In [ ]:
# [PATCHED] !pip install datasets

In [ ]:
from datasets import *

In [ ]:
train_ds = load_from_disk(working_folder  + 'train_dataset')
test_ds = load_from_disk(working_folder +'test_dataset')
val_ds = load_from_disk(working_folder +'val_dataset')

In [ ]:
train_ds

In [ ]:
model_id='google/vit-base-patch16-224-in21k'

In [ ]:
from transformers import ViTImageProcessor

In [ ]:
processor = ViTImageProcessor.from_pretrained(model_id)

In [ ]:
image=train_ds[0]['img']

type(image)

In [ ]:
import numpy as np

image_np=np.array(image, dtype=np.uint8)
image_np.shape

In [ ]:
image_np

In [ ]:
image_np_rgb=np.moveaxis(image_np, source=-1, destination=0)
image_np_rgb.shape

In [ ]:
processor_output=processor(image_np_rgb)
processor_output

In [ ]:
processor_output_np = np.array(processor_output['pixel_values'])
processor_output_np.shape

In [ ]:
def preprocess_images(dataset):

    images = dataset['img']

    images = [np.array(image, dtype=np.uint8) for image in images]
    images = [np.moveaxis(image, source=-1, destination=0) for image in images]

    processor_output = processor(images=images)

    dataset['pixel_values'] = processor_output['pixel_values']

    return dataset

In [ ]:
classes_names = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Sadness', 'Surprise', 'Neutral']

In [ ]:
features = Features({
    'label': ClassLabel(names=classes_names),
    'img': Array3D(dtype="int64", shape=(3,48,48)),
    'pixel_values': Array3D(dtype="float32", shape=(3, 224, 224)),
})

preprocessed_train_ds = train_ds.map(preprocess_images, batched=True,  features=features)

preprocessed_val_ds = val_ds.map(preprocess_images, batched=True, features=features)

preprocessed_test_ds = test_ds.map(preprocess_images, batched=True, features=features)

In [ ]:
preprocessed_train_ds

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
keys, counts = np.unique(preprocessed_train_ds["label"], return_counts=True)
plt.bar(classes_names, counts)
plt.show()

In [ ]:
preprocessed_train_ds.save_to_disk(working_folder  + 'preprocessed_train_dataset')
preprocessed_test_ds.save_to_disk(working_folder +'preprocessed_test_dataset')
preprocessed_val_ds.save_to_disk(working_folder + 'preprocessed_val_dataset')